# Structural Approach to Interventions

# 1. Kidney stone data simulation

In [ ]:
from numpy.random import uniform, seed
import pandas as pd

seed(1234)
patients_n = 10000


def g(size, u_1):
    # probabilities calculated with the conditional probability formula: P(T|S) = P(T, S)/P(S)
    # total patients = 750
    prob_small = 0.51
    prob_large = 1 - prob_small
    prob_A = (87 / 750) / prob_small if size == "S_0" else (263 / 750) / prob_large
    return "A" if u_1 < prob_A else "B"


def f(size, treatment, u_2):
    # probabilities obtained from Table "Recovery rates by treatment and size" in Chapter 2
    if size == "small":
        prob = 0.93 if treatment == "A" else 0.87
    else:
        prob = 0.73 if treatment == "A" else 0.62
    return 1 if u_2 < prob else 0


sizes = []
treatments = []
recoveries = []
for patient in range(patients_n):
    u_0 = uniform(size=1)
    u_1 = uniform(size=1)
    u_2 = uniform(size=1)

    size = "small" if u_0 < 0.51 else "large"
    treatment = g(size, u_1)
    recovery = f(size, treatment, u_2)

    sizes.append(size)
    treatments.append(treatment)
    recoveries.append(recovery)

kidney_data = pd.DataFrame(
    {"size": sizes, "treatment": treatments, "recovery": recoveries}
)
kidney_data.groupby("treatment")["recovery"].mean()

treatment
A    0.784981
B    0.809989
Name: recovery, dtype: float64

We obtain recovery rates similar to the Table “Recovery rates by
treatment” form Chapter 2 (up to some degree of uncertainty)

## 1.1 Intervening the treatment

If we give treatment A to everyone, we will not use the assignment
function `g`. Instead everyone will receive treatment A.

In [2]:
sizes = []
treatments = []
recoveries = []
for patient in range(patients_n):
    u_0 = uniform(size=1)
    u_1 = uniform(size=1)
    u_2 = uniform(size=1)

    size = "small" if u_0 < 0.51 else "large"
    treatment = "A"  # everyone receives treatment A
    recovery = f(size, treatment, u_2)

    sizes.append(size)
    treatments.append(treatment)
    recoveries.append(recovery)

kidney_data_A = pd.DataFrame(
    {"size": sizes, "treatment": treatments, "recovery": recoveries}
)
kidney_data_A.groupby("treatment")["recovery"].mean()

treatment
A    0.8328
Name: recovery, dtype: float64

We can see that the results are similar to the ones obtained from the
adjustment formula (up to some degree of uncertainty)

## 1.2 Randomizing the treament

If we run a randomized controlled trial, the treatment is given at
random

In [3]:
sizes = []
treatments = []
recoveries = []
for patient in range(patients_n):
    u_0 = uniform(size=1)
    u_1 = uniform(size=1)
    u_2 = uniform(size=1)

    size = "small" if u_0 < 0.51 else "large"
    treatment = "A" if u_1 < 0.5 else "B"  # treatment is assigned at random
    recovery = f(size, treatment, u_2)

    sizes.append(size)
    treatments.append(treatment)
    recoveries.append(recovery)

kidney_data_RCT = pd.DataFrame(
    {"size": sizes, "treatment": treatments, "recovery": recoveries}
)
kidney_data_RCT.groupby("treatment")["recovery"].mean()

treatment
A    0.833803
B    0.748213
Name: recovery, dtype: float64

In [3]:
! pip install matplotlib  pandas numpy seaborn statsmodels  

  Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (89 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached statsmodels-0.14.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.2 kB)
  Using cached contourpy-1.3.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.8-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.2 kB)
  Using cached pillow-11.2.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.9 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached patsy-1.0.1-py2.py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 3.0 MB/s eta 0:00:0000:0100:01
Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl (11.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [60]:
import pandas as pd

data = {
    ( 'A', 'Size_0', 0) : 6,
    ( 'A', 'Size_0', 1) : 81,
    ( 'A', 'Size_1', 0) : 71,
    ( 'A', 'Size_1', 1) : 192,
    ( 'B', 'Size_0', 0) : 36,
    ( 'B', 'Size_0', 1) : 234,
    ( 'B', 'Size_1', 0) : 30,
    ( 'B', 'Size_1', 1) : 50
}

collector = pd.DataFrame(columns = ['Treatment', 'Confounder', 'Result'])
for conf, nof_outcome in data.items():
    collector = pd.concat(
        [
            collector,
            pd.DataFrame(
                {
                    'Treatment': [conf[0]] * nof_outcome,
                    'Confounder': [conf[1]] * nof_outcome,
                    'Result': [conf[2]] * nof_outcome
                }
            )
        ],
        ignore_index=True
    ) 

In [85]:
data = collector.copy()

counts = ( data
    .groupby(['Confounder', 'Treatment'])
    .agg({'Result': 'count'})
   .reset_index()
   .pivot_table(
       index='Confounder',
       columns='Treatment',
       values='Result'
   )
).sum(axis =1)

totals = counts.sum()
perc = counts / totals

probs = ( data
    .groupby(['Confounder', 'Treatment'])
    .agg({'Result': 'mean'})
   .reset_index()
   .pivot_table(
       index='Confounder',
       columns='Treatment',
       values='Result'
   )
)

causal = probs.mul(perc, axis=0).sum()
causal




Treatment
A    0.832546
B     0.74825
dtype: object

In [153]:
data = collector.copy()

probs = ( data
    .groupby(['Confounder', 'Treatment'])
    .agg({'Result': 'mean'})
   .reset_index()
   .pivot_table(
       index='Confounder',
       columns='Treatment',
       values='Result'
   )
)

counts = ( data
    .groupby(['Confounder', 'Treatment'])
    .agg({'Result': 'count'})
   .reset_index()
   .pivot_table(
       index='Confounder',
       columns='Treatment',
       values='Result'
   )
)

effect_naive = data.groupby(['Treatment']).agg({'Result': 'mean'}).reset_index()

counts_causal = counts.sum(axis =1)
perc_causal = counts_causal / counts_causal.sum()
effekt_causal = probs.mul(perc_causal, axis=0).sum().reset_index()

print(f"Naive Effect:\n{effect_naive}\n\nCausal Effect:\n{effekt_causal}")

Naive Effect:
  Treatment    Result
0         A      0.78
1         B  0.811429

Causal Effect:
  Treatment         0
0         A  0.832546
1         B   0.74825


We can see that the results are similar to the ones obtained in the
previous execution and from the adjustment formula (up to some degree of
uncertainty)